# Ex.2: Hugging Face basics: tokenizers, pipelines, chat templates
**EECE.4860/5860**: ungraded practice for Lectures 2–3. Uses Qwen3-0.6B (Apache-2.0; no access request needed).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/exercises/ex2_hf_basics.ipynb)

In [ ]:
import importlib.util
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import ModelCard
from genai_utils import setup_env
device, dtype = setup_env()
MODEL = "Qwen/Qwen3-0.6B"

## 1. The model card
Every model on the Hub has a card: what it is, how it was trained, its license and intended use. Read the license before you use a model.

In [ ]:
card = ModelCard.load(MODEL)
print("license:", card.data.license)
print(card.text[:800], "...")

## 2. Tokens
The model never sees characters, only token ids.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
text = "Tokenization splits text into subword units."
ids = tok.encode(text)
print(ids)
print(tok.convert_ids_to_tokens(ids))
print("vocabulary size:", len(tok))

## 3. Chat templates
Chat models are trained on a specific conversation format. `apply_chat_template` builds it for you, including the special tokens. Qwen3 can also switch "thinking" on or off.

In [ ]:
msgs = [{"role": "user", "content": "Name three uses of generative AI."}]
for thinking in (False, True):
    print(f"--- enable_thinking={thinking} ---")
    print(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=thinking))

## 4. Generate
A `pipeline` hides tokenization, generation and decoding. Below we do the same steps by hand.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(device)
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
enc = tok(prompt, return_tensors="pt").to(device)
out = model.generate(**enc, max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.9)
print(tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
gen = pipeline("text-generation", model=model, tokenizer=tok)
print(gen(msgs, max_new_tokens=60, do_sample=False)[0]["generated_text"][-1]["content"])

## 5. Try it
* Change `temperature` to 0.1 and 1.5 and run cell 4 a few times. What changes?
* Print `model.generation_config`. Which sampling defaults did the model authors choose?
* Turn thinking on (`enable_thinking=True`) and look for the `<think>` block in the output.